# 🧬 MTase Topology Analyzer

**Automatic analysis and classification of DNA methyltransferases (MTases) based on beta-sheet topology**

---
## 📋 About

This tool analyzes 3D structures of DNA methyltransferases and automatically determines:
- Beta-strand order and orientation (↑/↓)
- Secondary structure elements (helices Hd/Hu)
- Catalytic motifs (NPPY, DPPY, GPPC, etc.)
- Structural class (A, B, C, D, E, F)

---
## 🚀 Option 1: Interactive Web App (Recommended)

The application is already deployed online. Click the badge below:

[![Open in Streamlit](https://static.streamlit.io/badges/streamlit_badge_black_white.svg)](https://mtasetopologyanalyser.streamlit.app/)

**No installation required.** The app will open in your browser.

### Features:
- Analyze by PDB ID, AlphaFold ID, or upload PDB file
- 3D structure visualization
- Automatic classification (A-F)
- Download results as CSV

---
## 💻 Option 2: Run Locally

If you prefer to run the app on your own computer:

```bash
git clone https://github.com/MVolobueva/MTase_topology_analyser.git
cd MTase_topology_analyser
pip install -r requirements.txt
streamlit run app.py
```

---
## 📊 Option 3: Single Structure Analysis (in Colab)

Run this cell to analyze one structure directly in the notebook.

In [ ]:
# @title Setup (Run First!)

import os

print("🚀 Setting up MTase Topology Analyzer...")
print("=" * 60)

# Clone repository
if not os.path.exists("MTase_topology_analyser"):
    !git clone https://github.com/MVolobueva/MTase_topology_analyser.git
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

os.chdir("MTase_topology_analyser")

# Install dependencies
print("\n📦 Installing Python dependencies...")
!pip install -q streamlit pandas numpy scipy biopython plotly py3dmol

print("\n✅ Setup complete!")
print("=" * 60)

In [0]:
# @title Run Single Structure Analysis

import sys
import io
import pandas as pd
from IPython.display import display, Markdown
from io import StringIO

def analyze_single_silent(id_value, source_type):
    """Run analysis with suppressed debug output"""
    
    old_stdout = sys.stdout
    sys.stdout = StringIO()
    
    try:
        from analyzer import MTaseAnalyzer
        from utils.helpers import download_structure
        from classifier import classify_topology
        import re
        
        print(f"\n🔬 Analyzing: {id_value} ({source_type})")
        
        if source_type == 'pdb':
            result_files = download_structure(id_value, source='pdb')
        elif source_type == 'alphafold':
            result_files = download_structure(id_value, source='alphafold (paste uniprot ID)')
        else:
            sys.stdout = old_stdout
            print("   Unsupported source type")
            return None
        
        if not result_files:
            sys.stdout = old_stdout
            print(f"   ❌ Failed to load {id_value}")
            return None
        
        analyzer = MTaseAnalyzer()
        if not analyzer.load_dssp(result_files['dssp']):
            sys.stdout = old_stdout
            print(f"   ❌ Failed to load DSSP")
            return None
        
        analyzer.find_all_strands()
        analyzer.build_sheet_adjacency()
        
        motifs = analyzer.find_all_motifs()
        motifs = analyzer.filter_motifs_by_topology(motifs)
        
        if not motifs:
            sys.stdout = old_stdout
            print(f"   ⚠️ No motifs found")
            return None
        
        results = []
        for motif_data in motifs:
            chain = motif_data['chain']
            motif_text = motif_data['text']
            motif_res = motif_data['res']
            motif_position = f"{motif_res}-{motif_res + len(motif_text) - 1}"
            
            result = analyzer.analyze_topology(motif_data=motif_data)
            if not result:
                continue
            
            out_buffer = StringIO()
            old_stdout2 = sys.stdout
            sys.stdout = out_buffer
            analyzer.print_linear_topology_from_result(result)
            sys.stdout = old_stdout2
            
            topology_str = out_buffer.getvalue()
            lines = topology_str.strip().split('\n')
            topology_line = ""
            for line in lines:
                if ('↑' in line or '↓' in line) and '[' in line and ']' in line:
                    topology_line = line.strip()
                    break
            
            if not topology_line:
                continue
            
            # Get full topology
            full_topology = topology_line
            
            # Get strands only
            strands_only = full_topology
            strands_only = re.sub(r'H[ud][_a-zA-Z0-9]*\[[\d-]+\]\s*—\s*', '', strands_only)
            strands_only = re.sub(r'^[Hh][ud][_a-zA-Z0-9]*\[[\d-]+\]\s*—\s*', '', strands_only)
            strands_only = re.sub(r'\s*—\s*[Hh][ud][_a-zA-Z0-9]*\[[\d-]+\]$', '', strands_only)
            
            # Get directions
            s_numbers = re.findall(r'S(-?\d+)\([↑↓]\)', full_topology)
            s_numbers_sorted = sorted(set(int(n) for n in s_numbers), reverse=True)
            directions = []
            for num in s_numbers_sorted:
                s_name = f"S{num}"
                match = re.search(f'{s_name}\([↑↓]\)', full_topology)
                if match:
                    if '(↑)' in match.group():
                        directions.append('↑')
                    elif '(↓)' in match.group():
                        directions.append('↓')
            directions_str = ''.join(directions)
            
            classification = classify_topology(full_topology, motif_text)
            
            results.append({
                'Chain': chain,
                'Motif': motif_text,
                'Position': motif_position,
                'Class': classification['class'],
                'Confidence': classification['confidence'],
                'Strand Order': classification['strand_order'],
                'Directions': directions_str,
                'Has S0': classification['has_s0'],
                'Has S-1': classification['has_s-1'],
                'Gap S6-S7': classification['gap_s6_s7']
            })
        
        sys.stdout = old_stdout
        
        if results:
            df = pd.DataFrame(results)
            display(Markdown(f"### 📊 Results for {id_value}"))
            display(df)
            return results
        else:
            print(f"   ⚠️ No valid topology found")
            return None
            
    except Exception as e:
        sys.stdout = old_stdout
        print(f"   ❌ Error: {str(e)}")
        return None

# @markdown ---
# @markdown Enter the structure ID below:
structure_id = "3S1S"  # @param {type:"string"}
structure_type = "pdb"  # @param ["pdb", "alphafold"]

analyze_single_silent(structure_id, structure_type)

---
## 📁 Option 4: Batch Analysis (Multiple Structures)

Upload a CSV file with multiple structures to analyze them all at once.

**CSV format:**
```csv
ID,Type
3S1S,pdb
2G1P,pdb
A0A7R8ZSU6,alphafold
```

Run the cell below, upload your CSV file, and download the results.

In [0]:
# @title Run Batch Analysis

from google.colab import files
import pandas as pd
import os
from IPython.display import display, Markdown

print("📁 Please upload your CSV file")
print("Expected format: ID,Type")
print("Example:")
print("  3S1S,pdb")
print("  A0A7R8ZSU6,alphafold")
print("-" * 50)

uploaded = files.upload()

for original_filename in uploaded.keys():
    print(f"\n📊 Processing: {original_filename}")
    
    !mv "{original_filename}" input.csv
    !python batch_analyze.py input.csv output.csv
    
    if os.path.exists('output.csv'):
        df = pd.read_csv('output.csv')
        display(Markdown(f"### 📊 Results for {original_filename}"))
        display(df)
        files.download('output.csv')
        print(f"\n✅ Results saved and downloaded")
    else:
        print(f"❌ Error: output.csv not created")

print("\n✅ Done!")

---
## 📖 Help & Documentation

### Output Columns

| Column | Description |
|--------|-------------|
| Chain | Chain identifier (A, B, C, etc.) |
| Motif | Catalytic motif (NPPY, DPPY, GPPC, etc.) |
| Class | Structural class (A/B/C/D/E/F) |
| Strand Order | Order of β-strands in N→C direction |
| Directions | Orientation arrows (↑ = antiparallel, ↓ = parallel) |
| Gap S6-S7 | Distance between strands S6 and S7 (indicates TRD insertion) |

### Classification Classes

| Class | Key Features |
|-------|--------------|
| **A** | S0 and S-1 at C-terminus |
| **B** | Permuted order (S7 at N-terminus) |
| **C** | C-helix only, reduced beta-sheet |
| **D** | Large gap S6-S7 (≥100 aa) |
| **E** | S0 at N-terminus, small gap |
| **F** | N-helix present |

### Links

- [GitHub Repository](https://github.com/MVolobueva/MTase_topology_analyser)
- [Report Issues](https://github.com/MVolobueva/MTase_topology_analyser/issues)

### Contact

M. Samokhina – mashila6799@gmail.com